In [ ]:
import json
import pandas as pd
import numpy as np

# Load the data
with open('output.json', 'r') as f:
    data = json.load(f)

predictions = data['predictions']

# List to store features for each frame
rows = []
prev_bbox_x = None

for pred in predictions:
    frame_id = pred['frame_id']
    bbox = pred['bbox'] # [x1, y1, x2, y2]
    curr_bbox_x = bbox[0]

    # Map keypoints for easy access
    kp_map = {k['class_name']: k for k in pred['keypoints']}

    def get_val(name, field='x'):
        return kp_map.get(name, {}).get(field, 0)

    # Features based on Research
    # 1. Eating: Nose lower than Neck
    nose_y = get_val('nose', 'y')
    neck_y = get_val('neck', 'y')
    nose_neck_diff = nose_y - neck_y

    # 2. Distress: Arched back (Neck vs Hip height)
    hip_y = get_val('hip', 'y')
    neck_hip_diff = neck_y - hip_y

    # 3. Walking: Front hoof to Back hoof stride
    f_hoof_x = get_val('left-front-hoof', 'x')
    b_hoof_x = get_val('left-back-hoof', 'x')
    stride = abs(f_hoof_x - b_hoof_x)

    # 4. Distress: Tail tuck (Tail X relative to Hip X)
    tail_x = get_val('tail', 'x')
    hip_x = get_val('hip', 'x')
    tail_tuck_dist = abs(tail_x - hip_x)

    # 5. Global Movement
    movement = 0
    if prev_bbox_x is not None:
        movement = abs(curr_bbox_x - prev_bbox_x)

    rows.append({
        'frame_id': frame_id,
        'nose_neck_y_diff': nose_neck_diff,
        'neck_hip_y_diff': neck_hip_diff,
        'stride_length': stride,
        'tail_hip_dist': tail_tuck_dist,
        'movement': movement,
        'nose_conf': kp_map.get('nose', {}).get('confidence', 0),
        'tail_conf': kp_map.get('tail', {}).get('confidence', 0)
    })

    prev_bbox_x = curr_bbox_x

df = pd.DataFrame(rows)

# Apply simple smoothing (rolling mean of 3 frames)
feature_cols = ['nose_neck_y_diff', 'neck_hip_y_diff', 'stride_length', 'tail_hip_dist', 'movement']
df[feature_cols] = df[feature_cols].rolling(window=3, min_periods=1).mean()

# Add a blank column for the user to label
df['behavior_label'] = ""

df.to_csv('new_video_behavior.csv', index=False)
print("CSV generated successfully with", len(df), "rows.")

CSV generated successfully with 490 rows.


In [ ]:
df = pd.read_csv("walk2_cow_behavior_features.csv")
df.head()

,frame_id,nose_neck_y_diff,neck_hip_y_diff,stride_length,tail_hip_dist,movement,nose_conf,tail_conf,behavior_label
0,3,-31.027527,60.497375,44.007202,0.350342,0.000000,0.656717,0.686016,NaN
1,4,-36.272995,75.985840,33.291626,6.434692,3.946838,0.553858,0.693541,NaN
2,5,-37.637166,78.470022,37.782267,8.899618,4.874329,0.569976,0.644099,NaN
3,6,-39.222432,84.924876,33.035156,13.571533,8.782043,0.527669,0.606114,NaN
4,7,-36.008016,78.564484,37.058065,12.940674,7.111694,0.561872,0.583187,NaN


In [ ]:
df['video_id'] = 9

In [ ]:
cols = ['video_id'] + [col for col in df.columns if col != 'video_id']
df = df[cols]

In [ ]:
df.head()

,frame_id,nose_neck_y_diff,neck_hip_y_diff,stride_length,tail_hip_dist,movement,nose_conf,tail_conf,behavior_label
0,3,-31.027527,60.497375,44.007202,0.350342,0.000000,0.656717,0.686016,NaN
1,4,-36.272995,75.985840,33.291626,6.434692,3.946838,0.553858,0.693541,NaN
2,5,-37.637166,78.470022,37.782267,8.899618,4.874329,0.569976,0.644099,NaN
3,6,-39.222432,84.924876,33.035156,13.571533,8.782043,0.527669,0.606114,NaN
4,7,-36.008016,78.564484,37.058065,12.940674,7.111694,0.561872,0.583187,NaN


# Manually Measuring frames

In [ ]:
import cv2

video_path = 'eat3.mp4'
cap = cv2.VideoCapture(video_path)
frame_id = 0

# Get video properties to save the output
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('labeled_reference_video.mp4', fourcc, fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Draw the Frame ID on the top-left corner
    cv2.putText(frame, f"Frame: {frame_id}", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
print("Reference video created: labeled_reference_video.mp4")

Reference video created: labeled_reference_video.mp4


# Random Forest

In [ ]:
import pandas as pd

# 1. Configuration
# FRAME_OFFSET = 3  # Automatically adds 3 to your input ranges
csv_file = 'walk2_cow_behavior_features.csv'

# 2. DEFINE YOUR RANGES (Using your 0-based video player numbers)
# Format: 'behavior_name': [(start, end), (start, end)]
behavior_ranges = {
    'walking':  [],                   # Label 0
    'eating':   [],           # Label 1
    'standing': [],            # Label 2
    'distress': [],                   # Label 3
    'lying':    []                    # Label 4
}

label_map = {
    'walking': 0, 'eating': 1, 'standing': 2, 'distress': 3, 'lying': 4
}

# 3. Load and Label
df = pd.read_csv(csv_file)

def get_label(frame_id):
    # Convert the CSV frame_id (starts at 3) back to 0-based for matching
    video_player_frame = frame_id

    for behavior, ranges in behavior_ranges.items():
        for start, end in ranges:
            if start <= video_player_frame <= end:
                return label_map[behavior]
    return 4

df['behavior_label'] = df['frame_id'].apply(get_label)

In [ ]:
df.tail()

,frame_id,nose_neck_y_diff,neck_hip_y_diff,stride_length,tail_hip_dist,movement,nose_conf,tail_conf,behavior_label
157,160,87.077484,95.954844,456.874634,73.993001,14.029968,0.649960,0.542035,NaN
158,161,72.738108,111.825348,442.048848,62.153361,25.392090,0.601288,0.516824,NaN
159,162,45.430827,121.925110,397.484945,48.233683,25.998520,0.657466,0.483359,NaN
160,163,33.147481,115.430115,360.953959,43.030843,21.067566,0.669261,0.505396,NaN
161,164,29.188599,104.639140,333.702759,42.256632,8.213013,0.554258,0.444332,NaN


In [ ]:
df['behavior_label'].unique()

array([0])

In [ ]:
df.to_csv('walk2_cow_behavior_features.csv', index=False)

# Training RF

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv("master_cow_data.csv")

In [ ]:
# frac=1 shuffles the entire dataset; random_state makes it reproducible
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
features = ['nose_neck_y_diff', 'neck_hip_y_diff', 'stride_length', 'tail_hip_dist', 'movement']
X = df_shuffled[features]
y = df_shuffled['behavior_label']

In [ ]:
# By default, train_test_split also shuffles, but shuffling the whole DF is safer
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Using 'balanced' weights to handle different class sizes
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
y_pred = rf_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Walking', 'Eating', 'Standing', 'Distress']))

              precision    recall  f1-score   support

     Walking       0.98      0.85      0.91        59
      Eating       0.94      0.99      0.97       133
    Standing       0.82      0.84      0.83        43
    Distress       1.00      1.00      1.00       327

    accuracy                           0.97       562
   macro avg       0.93      0.92      0.93       562
weighted avg       0.97      0.97      0.97       562



In [ ]:
import joblib
joblib.dump(rf_model, 'final_cow_behavior_model.pkl')

['final_cow_behavior_model.pkl']

# Evaluating on new video

In [ ]:
import joblib

# 1. Load the Model and the Scaler
model = joblib.load('final_cow_behavior_model.pkl')
scaler = joblib.load('cow_scaler.pkl')

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
new_video_df = pd.read_csv('new_video_behavior.csv')

In [ ]:
new_video_df.head()

,frame_id,nose_neck_y_diff,neck_hip_y_diff,stride_length,tail_hip_dist,movement,nose_conf,tail_conf,behavior_label
0,3,46.860260,102.408447,340.611023,29.460571,0.000000,0.347791,0.469971,NaN
1,4,43.552322,107.691895,340.165192,28.919739,0.481445,0.329266,0.470872,NaN
2,5,43.458242,108.153900,339.708130,30.497803,0.616435,0.323347,0.458157,NaN
3,6,42.188975,109.649882,339.329183,31.369059,0.936035,0.331852,0.476754,NaN
4,7,45.912730,106.524607,340.169759,31.919189,1.002991,0.353445,0.469627,NaN


In [ ]:
features = ['nose_neck_y_diff', 'neck_hip_y_diff', 'stride_length', 'tail_hip_dist', 'movement']
X_new = new_video_df[features]

In [ ]:
X_new_scaled = scaler.transform(X_new)

In [ ]:
new_video_df['behavior_label'] = model.predict(X_new_scaled)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
label_map = {0: 'Walking', 1: 'Eating', 2: 'Standing', 3: 'Distress', 4: 'Lying'}
new_video_df['behavior_name'] = new_video_df['behavior_label'].map(label_map)

In [ ]:
new_video_df.head()

,frame_id,nose_neck_y_diff,neck_hip_y_diff,stride_length,tail_hip_dist,movement,nose_conf,tail_conf,behavior_label,behavior_name
0,3,46.860260,102.408447,340.611023,29.460571,0.000000,0.347791,0.469971,1,Eating
1,4,43.552322,107.691895,340.165192,28.919739,0.481445,0.329266,0.470872,1,Eating
2,5,43.458242,108.153900,339.708130,30.497803,0.616435,0.323347,0.458157,1,Eating
3,6,42.188975,109.649882,339.329183,31.369059,0.936035,0.331852,0.476754,1,Eating
4,7,45.912730,106.524607,340.169759,31.919189,1.002991,0.353445,0.469627,1,Eating


In [ ]:
new_video_df.to_csv('new_video_results.csv', index=False)
print("Predictions complete! Check 'new_video_results.csv'")
print(new_video_df['behavior_name'].value_counts())

Predictions complete! Check 'new_video_results.csv'
behavior_name
Eating      272
Walking     188
Standing     24
Distress      6
Name: count, dtype: int64
